In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

In [30]:
ds = xr.open_dataset('solar_radiation2023janjune.nc')
ds = ds.sel(latitude=28.25, longitude=-16.75)
#ds = ds.sel(valid_time = slice('2023-04-13','2023-06-02'))
ds = ds.sel(valid_time = slice('2023-01-13','2023-02-23'))
ds

<xarray.Dataset> Size: 28kB
Dimensions:     (valid_time: 1008)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 8kB 2023-01-13 ... 2023-02-23T23:...
    number      int64 8B ...
    latitude    float64 8B 28.25
    longitude   float64 8B -16.75
    expver      (valid_time) <U4 16kB ...
Data variables:
    ssrd        (valid_time) float32 4kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-07-14T09:40 GRIB to CDM+CF via cfgrib-0.9.1...

In [31]:
df = ds.to_dataframe()
df.head

<bound method NDFrame.head of                          ssrd  number  latitude  longitude expver
valid_time                                                       
2023-01-13 00:00:00       0.0       0     28.25     -16.75   0001
2023-01-13 01:00:00       0.0       0     28.25     -16.75   0001
2023-01-13 02:00:00       0.0       0     28.25     -16.75   0001
2023-01-13 03:00:00       0.0       0     28.25     -16.75   0001
2023-01-13 04:00:00       0.0       0     28.25     -16.75   0001
...                       ...     ...       ...        ...    ...
2023-02-23 19:00:00  211456.0       0     28.25     -16.75   0001
2023-02-23 20:00:00       0.0       0     28.25     -16.75   0001
2023-02-23 21:00:00       0.0       0     28.25     -16.75   0001
2023-02-23 22:00:00       0.0       0     28.25     -16.75   0001
2023-02-23 23:00:00       0.0       0     28.25     -16.75   0001

[1008 rows x 5 columns]>

In [33]:

# --- 1. Convert J/m² (accumulated over 1 hour) to W/m² ---
df['ssrd_wm2'] = df['ssrd'] / 3600

# --- 2. Build the SECONDS column relative to the first timestamp ---
t0 = df.index[0]
seconds = (df.index - t0).total_seconds().astype(int)

# --- 3. Write the output file ---
initial = f"{t0.year}. {t0.month:>2}. {t0.day:>2}. {t0.hour}. {t0.minute}. {t0.second}."

start = df.index[0]
end = df.index[-1]
fileout = f'solar_radiation_{start.year}_{start.month}_{start.day}_{end.year}_{end.month}_{end.day}.dat'


with open(fileout, 'w') as f:
    f.write("TIME_UNITS                : SECONDS\n")
    f.write(f"SERIE_INITIAL_DATA        : {initial}\n")
    f.write("\n")
    f.write("SECONDS                   Solar radiation\n")
    f.write("<BeginTimeSerie>\n")
    for sec, val in zip(seconds, df['ssrd_wm2']):
        if pd.isna(val):
            continue
        f.write(f"{sec}                           {val:.5g}\n")
    f.write("<EndTimeSerie>\n")